# Trend Tracker — Notebook 01: Token Preprocess

SQL extract → clean, filtered token corpus. No analytical work happens here — this notebook is pure wrangling.

**Run order:** `01_token_preprocess` → `02_semantic_enrichment` → `03_insights_generation`

| Step | What | Key outputs |
|------|------|-------------|
| 1 | Ingest & attribute join | `prepared/01_ingested.parquet`, `prepared/02_project_essay_lookup.parquet` |
| 2 | Token normalization | `prepared/03_lemmatized.parquet` |
| 3 | Consolidation map build | `CONFIG/consolidation_map.csv`, `prepared/consolidation_audit.csv` |
| 4 | Consolidation apply | `prepared/04_consolidated.parquet` |
| 5 | Sparsity filter | `prepared/05_filtered.parquet`, `vocab/unigram_*.csv` |
| 6 | Quality checkpoint | `quality/quality_cp1.json` |

---
## Setup and Configuration

In [1]:
import sys
import yaml
from pathlib import Path

import pandas as pd

ROOT = Path(".")
sys.path.insert(0, str(ROOT))

import importlib.util as _ilu, sys as _sys
_u = next(Path(".").glob("utils*.py"))
if _u.stem != "utils":
    _s = _ilu.spec_from_file_location("utils", _u); _m = _ilu.module_from_spec(_s); _s.loader.exec_module(_m); _sys.modules["utils"] = _m

from utils import (
    load_cfg,
    flat_freq,
    token_doc_freq,
    normalize_tokens,
    build_consolidation_candidates,
    quality_report,
    build_output_path,
    ensure_warning_file,
    start_stage_manifest,
    finalize_stage_manifest,
    artifact_meta,
    resolve_params_path,
)

CFG_PATH = resolve_params_path()
CFG = load_cfg(CFG_PATH)

try:
    MANIFEST_CFG_PATH = str(CFG_PATH.relative_to(ROOT))
except ValueError:
    MANIFEST_CFG_PATH = str(CFG_PATH)

# Convenience wrapper: NB01 outputs are not run-scoped, so OUT() is a simple
# alias for build_output_path without groupby_field or run_date.
def OUT(subdir, fname):
    return build_output_path(subdir, fname)

# Warnings file — accumulates structured records for any step that
# encounters non-fatal data issues during this notebook run.
WARNINGS_PATH = OUT("prepared", "warnings_01.jsonl")
ensure_warning_file(WARNINGS_PATH)

STAGE_MANIFEST = start_stage_manifest(
    stage_name="01_preprocess",
    notebook_file="01_token_preprocess_v1.ipynb",
    config_path=MANIFEST_CFG_PATH,
    run_id=None,
    group_by_field=None,
    filter_fields_key=None,
)

# Stopword list for the quality gate at Step 6. Loaded from params.yaml so
# it can be extended without touching notebook code.
STOPWORDS = CFG["quality"]["stopword_violation_list"]

print(f"Warnings file  : {WARNINGS_PATH}")
print(f"Stopword count : {len(STOPWORDS)}")


Warnings file  : /Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/prepared/warnings_01.jsonl
Stopword count : 51


---
## Step 1 — Ingest

Reads all `DATA/project_essay*.csv` files via chunked reads, then left-joins
project attribute columns (category, grade band, school metadata) and derives
convenience date fields.

Two outputs are built in parallel:
- **Token corpus** (`df`) — the token lists that feed all downstream NLP work.
- **Essay text lookup** (`02_project_essay_lookup.parquet`) — raw essay text
  for snippet display in later steps only; does not affect topic modeling.

**Date authority:** `posted_date` and `funded_date` are read from the essay
extract files and treated as the authoritative source. Any date columns in
`project_attributes.csv` are dropped before the join to prevent duplicate
suffixes.

To add study-specific grouping columns (e.g. geographic cohorts, custom tiers),
join additional lookup CSVs in the attribute-join block below before saving
`01_ingested.parquet`.

In [2]:
DATA_DIR   = ROOT / "DATA"
CHUNK_SIZE = CFG["ingest"]["chunk_size"]

# ── Discover source files ─────────────────────────────────────────────────────
essay_files = sorted(DATA_DIR.glob("project_essay*.csv"))
if not essay_files:
    raise FileNotFoundError(f"No project_essay*.csv files found in {DATA_DIR}")
print(f"Source files: {[f.name for f in essay_files]}")

ESSAY_COLS          = ["project_id", "tokens", "posted_date", "funded_date"]
TEXT_COL_CANDIDATES = ["essay", "essay_text", "full_text", "project_essay", "text"]

chunks              = []  # token data — feeds the NLP pipeline
essay_lookup_chunks = []  # essay text — for snippet display only

# ── Chunked read ──────────────────────────────────────────────────────────────
# Two essay text sources are collected and ranked by source_priority:
#   priority 0 — dedicated text column (essay / essay_text / …)  ← preferred
#   priority 1 — reconstructed from LISTAGG token string         ← fallback
# Dedup below keeps priority-0 text for any project that has it, regardless
# of string length.
for fpath in essay_files:
    all_cols = pd.read_csv(fpath, nrows=0).columns.tolist()
    use_cols = [c for c in ESSAY_COLS if c in all_cols]
    text_col = next((c for c in TEXT_COL_CANDIDATES if c in all_cols), None)
    if text_col and text_col not in use_cols:
        use_cols = use_cols + [text_col]
    date_cols = [c for c in use_cols if c in {"posted_date", "funded_date"}]

    for chunk in pd.read_csv(
        fpath, chunksize=CHUNK_SIZE, usecols=use_cols,
        parse_dates=date_cols or False,
    ):
        # Fallback essay text: rejoin the LISTAGG token string with spaces.
        if "tokens" in chunk.columns:
            ec = (
                chunk[["project_id", "tokens"]]
                .rename(columns={"tokens": "essay_text"})
                .copy()
            )
            ec["essay_text"] = (
                ec["essay_text"].fillna("").astype(str)
                .str.replace(",", " ", regex=False)
                .str.replace(r"\s+", " ", regex=True).str.strip()
            )
            ec["source_priority"] = 1  # reconstructed — lower priority
            ec = ec[ec["essay_text"] != ""]
            if not ec.empty:
                essay_lookup_chunks.append(ec)

        # Parse LISTAGG string → token list for the NLP corpus.
        chunk["tokens"] = (
            chunk["tokens"].fillna("").str.split(",")
            .apply(lambda ts: [t.strip() for t in ts if t.strip()])
        )

        # Preferred essay text: dedicated text column when present.
        if text_col and text_col in chunk.columns:
            ec = (
                chunk[["project_id", text_col]]
                .rename(columns={text_col: "essay_text"})
                .copy()
            )
            ec["essay_text"] = (
                ec["essay_text"].fillna("").astype(str)
                .str.replace(r"\s+", " ", regex=True).str.strip()
            )
            ec["source_priority"] = 0  # dedicated text — higher priority
            ec = ec[ec["essay_text"] != ""]
            if not ec.empty:
                essay_lookup_chunks.append(ec)
            chunk = chunk.drop(columns=[text_col])

        chunks.append(chunk)

df = pd.concat(chunks, ignore_index=True)
print(f"Raw union: {len(df):,} rows across {len(essay_files)} file(s)")

# ── Build essay text lookup ───────────────────────────────────────────────────
# Sort by (project_id, source_priority asc, essay_len desc) so that the
# dedicated text column always wins over a reconstructed token string,
# regardless of length.
if essay_lookup_chunks:
    essay_lookup_df = (
        pd.concat(essay_lookup_chunks, ignore_index=True)
        .assign(essay_len=lambda x: x["essay_text"].str.len())
        .sort_values(
            ["project_id", "source_priority", "essay_len"],
            ascending=[True, True, False],
        )
        .drop_duplicates(subset=["project_id"], keep="first")
        .drop(columns=["essay_len", "source_priority"])
        .reset_index(drop=True)
    )
else:
    essay_lookup_df = pd.DataFrame(columns=["project_id", "essay_text"])

essay_lookup_path = OUT("prepared", "02_project_essay_lookup.parquet")
essay_lookup_df.to_parquet(essay_lookup_path, index=False)
print(f"Essay lookup: {len(essay_lookup_df):,} projects → {essay_lookup_path.name}")

# ── Left-join project attributes ──────────────────────────────────────────────
# The essay extract is the authoritative source for posted_date and funded_date.
# Drop those columns from attrs before merging to prevent _x / _y suffixes.
ATTR_CSV = DATA_DIR / "project_attributes.csv"
if not ATTR_CSV.exists():
    raise FileNotFoundError(f"project_attributes.csv not found in {DATA_DIR}")

attrs = pd.read_csv(ATTR_CSV)

# Only drop date columns from attrs if df already has them from the essay extract.
date_cols_to_drop = [
    c for c in ["posted_date", "funded_date"]
    if c in attrs.columns and c in df.columns
]
if date_cols_to_drop:
    attrs = attrs.drop(columns=date_cols_to_drop)

df = df.merge(attrs, on="project_id", how="left")
df["posted_date"]         = pd.to_datetime(df["posted_date"])
df["posted_year"]         = df["posted_date"].dt.year
df["posted_year_quarter"] = (
    df["posted_date"].dt.year.astype(str)
    + "_Q"
    + df["posted_date"].dt.quarter.astype(str)
)

# theme_version is a pipeline lineage field — increment when the token
# schema changes substantially across runs.
df["theme_version"] = "1.0"

df.to_parquet(OUT("prepared", "01_ingested.parquet"), index=False)
print(f"After attribute join: {len(df):,} rows  |  {len(df.columns)} columns")
print("Saved 01_ingested.parquet")


Source files: ['project_essays_01.csv', 'project_essays_02.csv', 'project_essays_03.csv', 'project_essays_04.csv', 'project_essays_05.csv', 'project_essays_06.csv', 'project_essays_07.csv', 'project_essays_08.csv', 'project_essays_09.csv', 'project_essays_10.csv', 'project_essays_11.csv', 'project_essays_12.csv', 'project_essays_13.csv', 'project_essays_14.csv', 'project_essays_15.csv', 'project_essays_16.csv', 'project_essays_17.csv', 'project_essays_18.csv', 'project_essays_19.csv', 'project_essays_20.csv', 'project_essays_21.csv', 'project_essays_22.csv', 'project_essays_23.csv', 'project_essays_24.csv', 'project_essays_25.csv']
Raw union: 2,239,480 rows across 25 file(s)
Essay lookup: 2,239,480 projects → 02_project_essay_lookup.parquet
After attribute join: 2,244,967 rows  |  45 columns
Saved 01_ingested.parquet


---
## Step 2 — Token Normalization

`normalize_tokens()` applies morphological normalization via `simplemma`,
collapsing inflected forms (`drives → drive`, `organized → organize`, `lives → live`).
Domain terms and proper nouns that `simplemma` does not recognise are returned
unchanged; Steps 3–4 handle any residual cases through the consolidation map.

In [3]:
df["tokens"] = df["tokens"].apply(normalize_tokens)
df.to_parquet(OUT("prepared", "03_lemmatized.parquet"), index=False)
print(f"Saved 03_lemmatized.parquet  |  {len(df):,} rows")


Saved 03_lemmatized.parquet  |  2,244,967 rows


---
## Steps 3 + 4 — Consolidation: Build Map & Apply

Builds a consolidation map automatically from the top-N vocabulary, then applies
it to `df` in one pass. No manual review step.

**Two automatic passes:**
1. **Known-fix dictionary** — corrects documented `simplemma` truncation artifacts
   (e.g. `creat → create`, `organiz → organize`). Loaded from
   `CONFIG/known_lemma_fixes.yaml`; extend that file to add new corrections.
2. **Plural-pair detection** — within the top-N vocab, maps `words → word` only
   when the singular form also appears in the same window.

**Outputs:**
- `CONFIG/consolidation_map.csv` — full candidate scan (audit trail)
- `OUTPUTS/prepared/consolidation_audit.csv` — applied mappings only
- `OUTPUTS/prepared/04_consolidated.parquet` — consolidated token corpus

In [4]:
# ── Load known lemma fixes ────────────────────────────────────────────────────
# Source: params.yaml → preprocess.known_lemma_fixes_path
# To add a new stem correction, edit CONFIG/known_lemma_fixes.yaml directly.
fixes_path = ROOT / CFG["preprocess"]["known_lemma_fixes_path"]
with open(fixes_path) as f:
    known_fixes = yaml.safe_load(f)
print(f"Known fixes: {len(known_fixes)} entries from {fixes_path.name}")

# ── Build consolidation candidates ───────────────────────────────────────────
# Scans the top-1000 vocabulary tokens, auto-assigns replacements via
# known_fixes and plural-pair detection, and flags likely truncation artifacts.
candidates = build_consolidation_candidates(df, known_fixes=known_fixes)

MAP_PATH = ROOT / "CONFIG" / "consolidation_map.csv"
MAP_PATH.parent.mkdir(parents=True, exist_ok=True)
candidates.to_csv(MAP_PATH, index=False)
print(
    f"{len(candidates)} tokens scanned  |  "
    f"{(candidates['replacement'] != '').sum()} auto-mapped  |  "
    f"{(candidates['flag_type'] == 'truncation').sum()} truncation-flagged"
)

# ── Apply map ─────────────────────────────────────────────────────────────────
cmap    = candidates[candidates["replacement"] != ""].copy()
mapping = dict(zip(cmap["original"].str.strip(), cmap["replacement"].str.strip()))
df["tokens"] = df["tokens"].apply(lambda ts: [mapping.get(t, t) for t in ts])

cmap[["original", "replacement", "flag_type", "notes"]].to_csv(
    OUT("prepared", "consolidation_audit.csv"), index=False
)
df.to_parquet(OUT("prepared", "04_consolidated.parquet"), index=False)

# Spot-check that key stems were actually remapped.
WATCH = {"creat", "engag", "experienc", "organiz", "includ", "writ", "inspir", "challeng"}
found = flat_freq(df)[lambda s: s.index.isin(WATCH)]
print(f"Mappings applied: {len(mapping)}")
print(
    "✓ No watched stems remain" if found.empty
    else f"⚠ Still present:\n{found.to_string()}"
)
candidates[candidates["replacement"] != ""].head(20)


Known fixes: 25 entries from known_lemma_fixes.yaml
1000 tokens scanned  |  6 auto-mapped  |  17 truncation-flagged
Mappings applied: 6
✓ No watched stems remain


,original,freq,flag_type,replacement,notes
93,develop,346339,,develop,auto
287,ipads,136159,,ipad,auto
317,means,127866,,mean,auto
608,towards,64158,,toward,auto
713,bags,53175,,bag,auto
846,whiteboards,43946,,whiteboard,auto


---
## Step 5 — Sparsity Filter

Applies corpus-level document-frequency bounds from `params.yaml`:
- **`sql.min_doc_count`** — drops tokens appearing in too few projects (noise)
- **`sql.max_doc_fraction`** — drops tokens appearing in too many projects (stopwords)
- **`sql.token_cap`** — caps tokens per project after filtering

Projects below `preprocess.min_project_tokens` after filtering are written to
`excluded_projects.parquet` and removed from the main corpus. Every retained
row is guaranteed to have passed all three thresholds.

When `preprocess.dedup_tokens_per_project` is true, each token is counted once
per project so TF-IDF reflects document presence rather than within-project frequency.

In [5]:
sq     = CFG["sql"]
sp_cfg = CFG["preprocess"]

# ── Compute document frequencies and keep-set ─────────────────────────────────
# doc_freq is computed on the full pre-exclusion corpus so that the keep-set
# reflects overall token prevalence, not a subset-dependent frequency.
doc_freq = token_doc_freq(df)
keep     = doc_freq[
    (doc_freq >= sq["min_doc_count"]) &
    (doc_freq <  len(df) * sq["max_doc_fraction"])
].index

# ── Filter tokens; optionally dedup within each project ───────────────────────
dedup = sp_cfg["dedup_tokens_per_project"]
df["tokens"] = df["tokens"].apply(
    lambda ts: list(dict.fromkeys(t for t in ts if t in keep))
    if dedup else [t for t in ts if t in keep]
)

# ── Apply per-project token cap ───────────────────────────────────────────────
df["tokens"] = df["tokens"].apply(lambda ts: ts[:sq["token_cap"]])

# ── Exclude under-tokenized projects ──────────────────────────────────────────
min_tok  = sp_cfg["min_project_tokens"]
excluded = df[df["tokens"].apply(len) < min_tok].copy()
excluded["exclusion_reason"] = f"< {min_tok} tokens after filtering"
df       = df[df["tokens"].apply(len) >= min_tok].reset_index(drop=True)

# ── Recompute doc frequencies on the final corpus ────────────────────────────
# Artifacts should describe 05_filtered.parquet, not the pre-exclusion state.
# Since all tokens in df are already in keep, final_doc_freq only contains
# retained-vocab tokens.
final_doc_freq = token_doc_freq(df)

# ── Persist ───────────────────────────────────────────────────────────────────
final_doc_freq.to_csv(OUT("vocab", "unigram_docfreq.csv"), header=True)
flat_freq(df).to_csv(OUT("vocab", "unigram_freq.csv"), header=True)
excluded.to_parquet(OUT("prepared", "excluded_projects.parquet"), index=False)
df.to_parquet(OUT("prepared", "05_filtered.parquet"), index=False)

print(f"Retained : {len(df):,}  |  Excluded : {len(excluded):,}  |  Vocab : {len(final_doc_freq):,}")


Retained : 2,244,916  |  Excluded : 51  |  Vocab : 7,554


---
## Step 6 — Quality Checkpoint 1

Gate for the SQL refinement loop. Review the output before proceeding to
`02_semantic_enrichment`:

- **No stopword violations** — if any terms from `quality.stopword_violation_list`
  appear in the top-200 vocab, tighten `sql.min_doc_count` or `sql.max_doc_fraction`
  in `params.yaml` and re-run from Step 1.
- **Reasonable token distribution** — check the `p50` tokens-per-project value.
  A healthy range after filtering is typically 20–60.
- **Vocabulary size** — should be in the low thousands; very large vocab may
  indicate the `min_doc_count` threshold is too permissive.

In [6]:
# Pass the configured stopword list so the gate reflects params.yaml rather
# than the HARD_STOPWORDS fallback in utils.py.
# Pass final_doc_freq so the checkpoint accurately describes 05_filtered.parquet.
qr1 = quality_report(
    df, label="cp1",
    doc_freq=final_doc_freq,
    save_path=OUT("quality", "quality_cp1.json"),
    stopwords=STOPWORDS,
)

# ── Stage manifest ────────────────────────────────────────────────────────────
stage_manifest_path = OUT("prepared/metadata", "stage_manifest_01_preprocess.json")
finalize_stage_manifest(
    STAGE_MANIFEST,
    output_path=stage_manifest_path,
    status="success",
    input_artifacts=[
        artifact_meta(DATA_DIR / "project_attributes.csv", "project_attributes_csv"),
        *[artifact_meta(p, f"essay_source::{p.name}") for p in essay_files],
    ],
    output_artifacts=[
        artifact_meta(OUT("prepared", "01_ingested.parquet"),             "ingested_parquet"),
        artifact_meta(OUT("prepared", "02_project_essay_lookup.parquet"), "project_essay_lookup_parquet"),
        artifact_meta(OUT("prepared", "03_lemmatized.parquet"),           "lemmatized_parquet"),
        artifact_meta(ROOT / "CONFIG" / "consolidation_map.csv",         "consolidation_map_csv"),
        artifact_meta(OUT("prepared", "consolidation_audit.csv"),         "consolidation_audit_csv"),
        artifact_meta(OUT("prepared", "04_consolidated.parquet"),         "consolidated_parquet"),
        artifact_meta(OUT("prepared", "05_filtered.parquet"),             "filtered_parquet"),
        artifact_meta(OUT("prepared", "excluded_projects.parquet"),       "excluded_projects_parquet"),
        artifact_meta(OUT("vocab",    "unigram_docfreq.csv"),             "unigram_docfreq_csv"),
        artifact_meta(OUT("vocab",    "unigram_freq.csv"),                "unigram_freq_csv"),
        artifact_meta(OUT("quality",  "quality_cp1.json"),                "quality_cp1_json"),
    ],
    row_counts={
        "input_projects":    int(len(df) + len(excluded)),
        "filtered_projects": int(len(df)),
        "excluded_projects": int(len(excluded)),
        "retained_vocab":    int(len(final_doc_freq)),
    },
    key_params={
        "min_doc_count":              sq["min_doc_count"],
        "max_doc_fraction":           sq["max_doc_fraction"],
        "min_project_tokens":         sp_cfg["min_project_tokens"],
        "token_cap":                  sq["token_cap"],
        "dedup_tokens_per_project":   sp_cfg["dedup_tokens_per_project"],
    },
    warnings_path=WARNINGS_PATH,
)
print(f"Stage manifest → {stage_manifest_path}")



=======================================================  [cp1]
  Projects : 2,244,916
  Tok/proj : min=5  p50=61  max=80
  Vocab    : 7,554 unique tokens
  Stopwords: FAIL — ['project', 'grade', 'teacher', 'education']

Stage manifest → /Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/prepared/metadata/stage_manifest_01_preprocess.json
